In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Receiver-Estimated-Align-V1: fixed Stage2

Only three saved G M05 RGB observations, with 181/177/89 received frames. Fresh framewise encoding once per observation and independent K0/K1 scoring use all 198 candidates and 4820 local rows. Only a complete finite unique canonical argmax estimates the offset; tie tolerance remains 1e-12 and sync_accepted remains false. An unresolved estimate retains its failed aligned slot without fallback. FULL singleton is geometry only.

The public receiver operation uses p=estimated_offset%4: prepend p copies of the first received frame, then drop the last p frames; p0 is identity and length stays fixed. Phases 2/3 are deterministic extensions, not validated by Stage1. New sync evidence is sealed before a physical phase plan is frozen. Baseline and aligned p0 share the same key read; same-phase keys share encoding but retain distinct key reads. No cross-observation cache. 12 logical payload slots retain 422400 votes, 14080 time-bit rows and 384 final bits; physical upper bounds are 7 Wan encodes and 10 reads (aliases are not independent observations). Wan FP32 posterior mode and the original R44/R44/R22 reader, coordinates, strict >0 and Counter encounter-order ties remain unchanged.

Actual truth and historical reference reads occur only after the final payload blind seal. All failures, wrong-key results, ties and full time/channel evidence remain. Historical comparisons cannot select offsets, phases, retries or a better result. Receiver latent indices do not assert matched or independent source receptive fields. No source generation, writer, codec, quality recomputation, strength scan, thresholds, FPR claim or production integration. This fixed test does not establish generalized synchronization, cache recovery or unique phase causality.

The notebook is user-run only after reviewed immutable source publication. Local checks are CPU/fake/static only. Relevant eight package pins reuse the successful receiver environment; actual versions and pip-check warnings are preserved.

Fixed received RGB identities:
- obs_83bd426a52f36e63: /content/drive/MyDrive/Video-WM/Trajectory-Payload-G-V1/20261005T140121854219Z/fixed_reference/M05_FRAMEWISE_SYNC/received.rgb8; SHA256 ce1c6346ddfeb852d57f21d82d6f4452bc491d7d1a73a688108f4aff13683295
- obs_9d4a3f27ce933f70: /content/drive/MyDrive/Video-WM/Trajectory-Payload-G-V1/20261005T140121854219Z/fixed_reference/M05_FRAMEWISE_SYNC/received.CROP177.rgb8; SHA256 5b03066d2e4555dd74d59843ed78971762758d422703153c824be9c1a92f0984
- obs_30cdbf6917442b0c: /content/drive/MyDrive/Video-WM/Trajectory-Payload-G-V1/20261005T140121854219Z/fixed_reference/M05_FRAMEWISE_SYNC/received.SHORT89.rgb8; SHA256 1cff184d33b9b18e2d5d612a0792e6564a1c201a39d676483e5d050031845531

Published source: 079c766ac910961492c5e50c8386adbf3e2eb55f

In [ ]:
SOURCE_SHA = '079c766ac910961492c5e50c8386adbf3e2eb55f'
FIXED = {'observations': 3, 'sync_reads': 6, 'sync_candidate_scores': 198, 'sync_candidate_tubelet_rows': 4820, 'logical_payload_reads': 12, 'logical_detailed_votes': 422400, 'logical_time_bit_rows': 14080, 'logical_final_bit_rows': 384}
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Trajectory-Receiver-Estimated-Align-Stage2-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-TRAJECTORY-RECEIVER-ESTIMATED-ALIGN-STAGE2-V1-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    probe="import importlib.metadata as m; pins={'torch': '2.11.0', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'numpy': '2.1.3', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2'}; actual={k:m.version(k) for k in pins}; assert all(actual[k].split('+')[0]==v for k,v in pins.items()), actual; from diffusers import AutoencoderKL, AutoencoderKLWan; import torch; print(actual)"
    if logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','torch==2.11.0','diffusers==0.39.0','transformers==4.57.6','numpy==2.1.3','accelerate==1.15.0','safetensors==0.8.0','huggingface-hub==0.36.2','tokenizers==0.22.2'],'RECEIVER_DEPENDENCY_REPAIR')
        logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_REPROBE')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:pass
finally:token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_trajectory_receiver_estimated_align_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'],'Calls:',result['calls'])
print('Physical plan:',result['physical_plan']['planned_calls'],'logical aliases are not independent evidence')
print('Actual environment:',result['environment'])
print('Seals:',result.get('blind_sync_sha256'),result.get('physical_plan_sha256'),result.get('blind_payload_sha256'))
for sid,row in sorted(result['estimates'].items()):
    print('Blind estimate:',sid,row)
for sid,row in sorted(result['sync_posthoc'].items()):
    print('Postseal sync:',sid,'key=',row.get('key_role'),'truth_offset=',row.get('truth_offset'),
          'estimated_offset=',row.get('estimated_offset'),'offset_correct=',row.get('offset_correct'),
          'phase_correct=',row.get('phase_correct'),'geometry_only=',row.get('geometry_only'))
for sid,row in sorted(result['payload_posthoc'].items()):
    values=[x['signed_normalized_margin'] for x in row.get('bit_rows',[]) if 'signed_normalized_margin' in x]
    print(sid,row['status'],'key=',row.get('key_role'),'view=',row.get('view'),'phase=',row.get('phase'),
          'alias=',row.get('alias_of'),'bit_errors=',row.get('bit_errors'),'error_bits=',row.get('error_bits'),'minimum signed margin=',min(values,default=None))
    print('  full detailed evidence:',result['payload_reads'][sid]['detail_path'],row.get('path'))
for sid,row in sorted(result['same_run_differences'].items()):
    print('EST_ALIGN minus BASELINE:',sid,row['status'],'bit error delta=',row.get('bit_error_delta'))
for sid,row in sorted(result['historical_comparisons'].items()):
    print('Historical postseal comparison:',sid,row)
print('Failures:',result['failures'])
print('Evidence ceiling:',result['evidence_ceiling'])
print('Result:',RESULT_PATH)
